# Set up in cloud

### For Colab notebooks, start here

In [ ]:
!git clone https://${GITHUB_TOKEN}@github.com/getsentry/grouping-trainer.git

In [ ]:
%cd grouping-trainer

### For Workbench notebooks, start here

In [1]:
!git rev-parse --short HEAD

5d78b2b


After running this `pip install` cell, restart the notebook session. TODO: activate venv instead

In [3]:
!pip install -e .

Obtaining file:///home/jupyter/grouping-trainer
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for grouping-trainer (pyproject.toml) ... done
  Created wheel for grouping-trainer: filename=grouping_trainer-0.1.0-0.editable-py3-none-any.whl size=10668 sha256=5a55ab81038fee7934f792ff5ad77c51c8b32e0e810d93546335f0677beb50a5
  Stored in directory: /var/tmp/pip-ephem-wheel-cache-0y3ess5k/wheels/5f/ca/75/881118fa97e0f695c6b005134dc42a54df5074c95ebd09bef1
Successfully built grouping-trainer
  Attempting uninstall: grouping-trainer
    Found existing installation: grouping-trainer 0.1.0
    Uninstalling grouping-trainer-0.1.0:
      Successfully uninstalled grouping-trainer-0.1.0


In [4]:
!gsutil -m cp -r gs://seer-models/models/issue_grouping_v1 .

Copying gs://seer-models/models/issue_grouping_v1/.DS_Store...
Copying gs://seer-models/models/issue_grouping_v1/data.pkl...                   
Copying gs://seer-models/models/issue_grouping_v1/embeddings/1_Pooling/config.json...
Copying gs://seer-models/models/issue_grouping_v1/embeddings/README.md...       
Copying gs://seer-models/models/issue_grouping_v1/embeddings/config.json...     
Copying gs://seer-models/models/issue_grouping_v1/embeddings/model.safetensors...
Copying gs://seer-models/models/issue_grouping_v1/embeddings/vocab.json...
Copying gs://seer-models/models/issue_grouping_v1/embeddings/config_sentence_transformers.json...
Copying gs://seer-models/models/issue_grouping_v1/embeddings/configuration_bert.py...
Copying gs://seer-models/models/issue_grouping_v1/embeddings/merges.txt...      
Copying gs://seer-models/models/issue_grouping_v1/embeddings/modules.json...    
Copying gs://seer-models/models/issue_grouping_v1/embeddings/modeling_bert.py...
Copying gs://seer-models

In [5]:
!mkdir gte-finetuned
!gsutil -m cp -r gs://grouping-data/runs/./2025-12-19-15-53-06-gte-output/training gte-finetuned/

Copying gs://grouping-data/runs/./2025-12-19-15-53-06-gte-output/training/1_Pooling/config.json...
Copying gs://grouping-data/runs/./2025-12-19-15-53-06-gte-output/training/README.md...
Copying gs://grouping-data/runs/./2025-12-19-15-53-06-gte-output/training/checkpoint-3300/config.json...
Copying gs://grouping-data/runs/./2025-12-19-15-53-06-gte-output/training/checkpoint-3300/1_Pooling/config.json...
Copying gs://grouping-data/runs/./2025-12-19-15-53-06-gte-output/training/checkpoint-3300/README.md...
Copying gs://grouping-data/runs/./2025-12-19-15-53-06-gte-output/training/checkpoint-3300/config_sentence_transformers.json...
Copying gs://grouping-data/runs/./2025-12-19-15-53-06-gte-output/training/checkpoint-3300/model.safetensors...
Copying gs://grouping-data/runs/./2025-12-19-15-53-06-gte-output/training/checkpoint-3300/modules.json...
Copying gs://grouping-data/runs/./2025-12-19-15-53-06-gte-output/training/checkpoint-3300/scheduler.pt...
Copying gs://grouping-data/runs/./2025-12

In [6]:
!gsutil -m -o GSUtil:check_hashes=never cp -r gs://grouping-data/final_csvs .

Copying gs://grouping-data/final_csvs/sentry.csv...
Copying gs://grouping-data/final_csvs/synthetic-semi-easy-negatives.csv...      
Copying gs://grouping-data/final_csvs/test.csv...                               
Copying gs://grouping-data/final_csvs/train.csv...                              
Copying gs://grouping-data/final_csvs/train_no_sentry.csv...                    
Copying gs://grouping-data/final_csvs/val.csv...
\ [6/7 files][  9.0 GiB/  9.0 GiB]  99% Done 192.1 MiB/s ETA 00:00:00           

# Run

In [1]:
import os
import json
from datetime import datetime
import time

import numpy as np
import polars as pl
from pydantic import BaseModel, field_serializer
from sentence_transformers.util import pairwise_cos_sim
import torch
from tqdm.auto import tqdm

import grouping_trainer as gt
import utils

`torch` gives this helpful warning w/o this next line:

```
/opt/conda/lib/python3.10/site-packages/torch/_inductor/compile_fx.py:282: UserWarning: TensorFloat32 tensor cores for float32 matrix multiplication available but not enabled. Consider setting `torch.set_float32_matmul_precision('high')` for better performance.
```

In [2]:
torch.set_float32_matmul_precision('high')  # no impact it seems?

In [3]:
class ModelConfig(BaseModel):
    name: str
    path: str
    truncate_dim: int | None = None
    batch_size: int = 1
    model_kwargs: dict | None = None

    @field_serializer("model_kwargs")
    def serialize_model_kwargs(self, v: dict | None) -> dict:
        if v is None:
            return None
        return {k: str(val) if isinstance(val, torch.dtype) else val for k, val in v.items()}


class ModelConfigs(BaseModel):
    model_configs: list[ModelConfig]


class DataConfig(BaseModel):
    val_df_path: str
    sample_size: int | None = None

In [4]:
timestamp = datetime.now().strftime("%Y-%m-%d-%H-%M-%S")

In [5]:
RUN_SHORTNAME = "val"
DATA_CONFIG = DataConfig(
    val_df_path="final_csvs/val.csv",
    sample_size=None,
    # sample_size=100,
)
MODEL_CONFIGS = ModelConfigs(
    model_configs=[
        ModelConfig(
            name="gte-finetuned",
            path="gte-finetuned/training",
            truncate_dim=64,
            model_kwargs=dict(
                dtype=torch.bfloat16,
                attn_implementation="sdpa",
                # attn_implementation="flash_attention_2",  # hell
            ),
        ),
        ModelConfig(
            name="prod",
            path="issue_grouping_v1/embeddings",
            truncate_dim=None,
        ),
    ]
)
OUTPUT_DIR = f"./{timestamp}-{RUN_SHORTNAME}"

In [6]:
def encode_timed(
    model: gt.utils.SentenceTransformer, texts: list[str], progress_bar_desc: str | None = None
) -> tuple[np.ndarray, list[float]]:
    times: list[float] = []
    embeddings: list[np.ndarray] = []
    for text in tqdm(texts, desc=progress_bar_desc):
        start = time.monotonic()
        emb = model.encode(text, convert_to_numpy=True)
        end = time.monotonic()
        times.append(end - start)
        embeddings.append(emb)
    return np.array(embeddings), times

In [7]:
df = utils.load_val_df(path=DATA_CONFIG.val_df_path, sample_size=DATA_CONFIG.sample_size)
print(df.shape)
print(df.columns)

(84995, 28)
['query_seer_event_sent', 'candidate_seer_event_sent', 'distance', 'query_group_id', 'candidate_group_id', 'query_hash', 'candidate_hash', 'query_grouphash_id', 'candidate_grouphash_id', 'query_grouphashmetadata_id', 'candidate_grouphashmetadata_id', 'query_seer_gr_id', 'candidate_seer_gr_id', 'query_error_type', 'candidate_error_type', 'project_id', 'platform', 'source', 'path', 'query_stacktrace_string', 'candidate_stacktrace_string', 'label', 'thinking_output', 'response_output', 'confidence_score', 'prompt', 'org_id', 'is_grouped']


In [8]:
for model_config in tqdm(MODEL_CONFIGS.model_configs, desc="Models"):
    print(model_config)

    if model_config.model_kwargs:
        st_class = gt.danger.SentenceTransformer
    else:
        st_class = gt.utils.SentenceTransformer

    model = st_class(
        model_config.path,
        trust_remote_code=True,
        truncate_dim=model_config.truncate_dim,
        model_kwargs=model_config.model_kwargs,
    )

    start = time.monotonic()
    if hasattr(model, "warmup_and_compile"):
        model.warmup_and_compile()
    else:
        _ = model.encode("warm up")
    end = time.monotonic()
    warm_up_time = round(end - start, 1)
    print(f"Warm up took {warm_up_time} seconds.")

    query_texts = df["query_stacktrace_string"].to_list()
    query_embeddings, query_times = encode_timed(
        model, query_texts, progress_bar_desc="Queries"
    )

    candidate_texts = df["candidate_stacktrace_string"].to_list()
    candidate_embeddings, candidate_times = encode_timed(
        model, candidate_texts, progress_bar_desc="Candidates"
    )

    cos_sims = pairwise_cos_sim(query_embeddings, candidate_embeddings).detach().cpu().numpy()

    df = df.with_columns(
        [
            pl.Series(name=f"cos_sim_{model_config.name}", values=cos_sims),
            pl.Series(name=f"query_encode_time_{model_config.name}", values=query_times),
            pl.Series(name=f"candidate_encode_time_{model_config.name}", values=candidate_times),
        ]
    )

Models:   0%|          | 0/2 [00:00<?, ?it/s]

name='gte-finetuned' path='gte-finetuned/training' truncate_dim=64 batch_size=1 model_kwargs={'dtype': torch.bfloat16, 'attn_implementation': 'sdpa'}


W0219 01:55:20.797000 50867 site-packages/torch/fx/experimental/symbolic_shapes.py:6823] [0/1] _maybe_guard_rel() was called on non-relation expression Eq(s11, 1) | Eq(s36, s11)


Warm up took 13.7 seconds.


Queries:   0%|          | 0/84995 [00:00<?, ?it/s]

...

name='prod' path='issue_grouping_v1/embeddings' truncate_dim=None batch_size=1 model_kwargs=None


/opt/conda/lib/python3.10/site-packages/torch/onnx/_internal/registration.py:162: OnnxExporterWarning: Symbolic function 'aten::scaled_dot_product_attention' already registered for opset 14. Replacing the existing function with new function. This is unexpected. Please report it on https://github.com/pytorch/pytorch/issues.
  warnings.warn(


Warm up took 0.1 seconds.


Queries:   0%|          | 0/84995 [00:00<?, ?it/s]

Candidates:   0%|          | 0/84995 [00:00<?, ?it/s]

# Upload

In [9]:
os.mkdir(OUTPUT_DIR)

In [10]:
with open(f"{OUTPUT_DIR}/model_configs.json", "w") as f:
    json.dump(MODEL_CONFIGS.model_dump(), f, indent=4)

with open(f"{OUTPUT_DIR}/data_config.json", "w") as f:
    json.dump(DATA_CONFIG.model_dump(), f, indent=4)

In [11]:
df.write_csv(f"{OUTPUT_DIR}/similarities.csv")

In [12]:
!gsutil -m rsync -r {OUTPUT_DIR} gs://grouping-data/similarities/{OUTPUT_DIR}

Building synchronization state...
Starting synchronization...
Copying file://./2026-02-19-01-55-09-val/data_config.json [Content-Type=application/json]...
Copying file://./2026-02-19-01-55-09-val/similarities.csv [Content-Type=text/csv]...
==> NOTE: You are uploading one or more large file(s), which would run          
significantly faster if you enable parallel composite uploads. This
feature can be enabled by editing the
"parallel_composite_upload_threshold" value in your .boto
configuration file. However, note that if you do this large files will
be uploaded as `composite objects
<https://cloud.google.com/storage/docs/composite-objects>`_,which
means that any user who downloads such objects will need to have a
compiled crcmod installed (see "gsutil help crcmod"). This is because
without a compiled crcmod, computing checksums on composite objects is
so slow that gsutil disables downloads of composite objects.

Copying file://./2026-02-19-01-55-09-val/model_configs.json [Content-Type=

In [13]:
!gsutil -m cp -r run.ipynb gs://grouping-data/similarities/{OUTPUT_DIR}

Copying file://run.ipynb [Content-Type=application/octet-stream]...
/ [1/1 files][ 45.3 KiB/ 45.3 KiB] 100% Done                                    
Operation completed over 1 objects/45.3 KiB.                                     
